# Fig 4E / 4F — full GTEx approximated with ADSP + UKB (data processing)

Full GTEx summary statistics (n = 838) fine-mapped with the ADSP sketch, the UKB sketch and their estimated mixture (`27.Fig4E_4F.GTEx_ADSP_UKB_Mix`: one `susie_rss` call on the list of sketches). For each LD input, SuSiE-RSS and SuSiE-rRSS; SuSiE-rRSS falls back to its SER model when `R_reliability_flag` is set (as in Fig S3E).

**Outputs:**
- `figure_data/Fig4F.GTEx_Mix_Power_FDR.rds` — power / FDR by LD input × method (bootstrap 95% CI)
- `figure_data/Fig4E.GTEx_ADSP_Weight.rds` — estimated ADSP weight of the SuSiE-RSS mixture fit, one value per simulation

In [1]:
source("../../R/utils.R")

res_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_ADSP_UKB_mix/summary/result_mix_summary.rds"
n_GWAS   <- 800      # nominal; nh2 = 800 x h2g = 40 / 80 / 120

## Fig 4E: power / FDR

In [2]:
res <- readRDS(res_file) %>%
  mutate(h2g = str_extract(input_file, "(?<=h2g_)[0-9.]+"), causal = ncausal)

ld_lab <- c(mix = "Estimated mixture", ADSP = "ADSP", UKB = "UK Biobank")
keys   <- c("input_file", "ncausal")

## SuSiE-RSS: the plain fits
rss <- res %>%
  filter(method %in% names(ld_lab)) %>%
  mutate(ld_reference = ld_lab[method], analysis_method = "SuSiE-RSS")

## SuSiE-rRSS: the *_rrss fit, or its SER fallback model when R_reliability_flag is set
rrss <- map_dfr(names(ld_lab), function(m) {
  fit <- filter(res, method == paste0(m, "_rrss"))
  ser <- filter(res, method == paste0(m, "_rrss_SER"))
  use_ser <- fit %>% filter(R_reliability_flag %in% TRUE) %>% select(all_of(keys)) %>% semi_join(ser, by = keys)
  bind_rows(ser %>% semi_join(use_ser, by = keys),
            fit %>% anti_join(use_ser, by = keys)) %>%
    mutate(ld_reference = ld_lab[m], analysis_method = "SuSiE-rRSS")
})

## SER fallback frequency per LD input (for the text)
res %>% filter(method %in% paste0(names(ld_lab), "_rrss")) %>%
  group_by(method) %>%
  summarise(n = n(), n_fallback = sum(R_reliability_flag %in% TRUE), pct = round(100 * mean(R_reliability_flag %in% TRUE), 1))

## bootstrap 95% CIs (boot_cell_mix() in R/utils.R)
n_boot <- 1000
set.seed(1)
summary_ci <- bind_rows(rss, rrss) %>%
  group_by(ld_reference, analysis_method, h2g, causal) %>%
  group_modify(~ boot_cell_mix(.x, B = n_boot)) %>%
  ungroup()

pl <- summary_ci %>%
  mutate(ncp_num = n_GWAS * parse_h2g(h2g),
         ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 0)), ncp_num)) %>%
  transmute(ld_reference, analysis_method, ncp, causal,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est   = FDR,   FDR_lo   = FDR_lo,   FDR_hi   = FDR_hi) %>%
  pivot_longer(-c(ld_reference, analysis_method, ncp, causal),
               names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(
    metric          = factor(metric, levels = c("Power", "FDR")),
    causal          = factor(paste0(causal, " causal"),
                             levels = paste0(sort(unique(as.integer(causal))), " causal")),
    ld_reference    = factor(ld_reference, levels = unname(ld_lab)),
    analysis_method = factor(analysis_method, levels = c("SuSiE-RSS", "SuSiE-rRSS")),
    is_placeholder  = FALSE
  )
stopifnot(!any(is.na(pl$ld_reference)), nrow(pl) == 108)

save_data(pl, "Fig4F.GTEx_Mix_Power_FDR")

## FDR / power ranges across settings (for the text)
pl %>% group_by(analysis_method, ld_reference, metric) %>%
  summarise(min = round(100 * min(est), 1), max = round(100 * max(est), 1), .groups = "drop") %>%
  arrange(metric, analysis_method, ld_reference)

method,n,n_fallback,pct
<chr>,<int>,<int>,<dbl>
ADSP_rrss,2536,40,1.6
UKB_rrss,2536,136,5.4
mix_rrss,2536,40,1.6


analysis_method,ld_reference,metric,min,max
<fct>,<fct>,<fct>,<dbl>,<dbl>
SuSiE-RSS,Estimated mixture,Power,92.0,99.6
SuSiE-RSS,ADSP,Power,91.8,99.6
SuSiE-RSS,UK Biobank,Power,92.2,99.3
SuSiE-rRSS,Estimated mixture,Power,89.7,99.6
SuSiE-rRSS,ADSP,Power,89.4,99.6
SuSiE-rRSS,UK Biobank,Power,83.5,99.6
SuSiE-RSS,Estimated mixture,FDR,3.8,21.5
SuSiE-RSS,ADSP,FDR,3.5,21.2
SuSiE-RSS,UK Biobank,FDR,4.2,26.0


## Fig 4F: estimated ADSP weight

In [3]:
## one estimate per simulation (all NCP pooled), from the SuSiE-RSS mixture fit
loci <- readRDS(res_file) %>%
  distinct(input_file, .keep_all = TRUE) %>%
  transmute(weight = ADSP_weight, weight_rrss = ADSP_weight_rrss) %>%
  filter(!is.na(weight))

quantile(loci$weight, c(.25, .5, .75))     # median / IQR quoted in the text

save_data(loci, "Fig4E.GTEx_ADSP_Weight")

25%       50%       75% 
0.6814017 0.7459426 0.7500000